In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import AdamW

In [2]:
from datasets import load_dataset

dataset = load_dataset("text", data_files={"train": "data/output_chat.txt"})

In [3]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])

In [4]:
from transformers import pipeline, set_seed

model = GPT2LMHeadModel.from_pretrained("gpt2") # 124M
hf_generator = pipeline('text-generation', model='gpt2')
model.resize_token_embeddings(len(tokenizer))  # In case vocab was extended

Device set to use cuda:0


Embedding(50257, 768)

In [5]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./models/gpt2-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
    prediction_loss_only=True,
    logging_steps=100,
    fp16=True,  # if using GPU
    evaluation_strategy="no",
)


/home/nitin/miniconda3/envs/dl/lib/python3.12/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [6]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False  # Causal LM, not MLM
)

In [8]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()


/tmp/ipykernel_37343/737994221.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,4.064800
200,3.495600
300,3.466800
400,3.297300
500,3.212300
600,3.266700
700,3.142000
800,3.000200
900,3.102700
1000,3.018500


TrainOutput(global_step=6435, training_loss=2.695039126997135, metrics={'train_runtime': 785.7343, 'train_samples_per_second': 32.752, 'train_steps_per_second': 8.19, 'total_flos': 6724089151488000.0, 'train_loss': 2.695039126997135, 'epoch': 3.0})

In [ ]:
output_dir = "./models/gpt2-finetuned"

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

In [15]:
output_dir = "./models/gpt2-finetuned/checkpoint-6435"
model = GPT2LMHeadModel.from_pretrained(output_dir)

In [18]:
from transformers import pipeline

# Use Hugging Face pipeline for text generation
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

prompt = "Nitin:"
output = generator(prompt, max_length=500, do_sample=True, temperature=0.9)

print(output[0]["generated_text"])

Device set to use cuda:0


Nitin: Ye fobo. Meri kara lo mujhe. Nikalunga. Baking up karo kyuki liya tha?. Kha lo. Isko nahi koi mai office me hogi kya. 🌝🌝🌝🌝🌝🌝🌝🌝🌝🌝🌝🌝🌝🌝🌝🌝. Nitin: utha rahi. Wifey <3: utha rahi utha rahi. Wifey <3: utha rahi utha rahi. Wifey <3: utha rahi siii. Wifey <3: utha rahi siii. Wifey <3: utha rahi siii. Wifey <3: utha rahi siii. Wifey <3: utha rahi siii. Wifey <3: utha rife hai. Wifey <3: utha rahi mai. Wifey <3: utha rahi siii. Wifey <3: utha rahi siii. Wifey <3: utha rahi siii. Wifey <3: utha rahi siii. Wifey <3: utha rahi siii. Wifey <3: utha rahi siii. Wifey <3: utha rahi siii. Wifey <3: utha rahi sii. Wifey <3: utha rahi siii. Wifey <3: utha rahi sii. Wifey <3: utha rahi sii. Wifey <3: utha rahi siii. Wifey <3: utha rahi siii. Wifey <3: utha rahi siii. Wifey <3: utha rahi siii. Wifey <3: utha rahi siii. Wifey <3: utha rehne hi thodi kya. Wifey <3: utha rehne hi thodi kya. Wifey <3: utha rehne hi thodi kiya. Wifey <3: uth
